# MCTS Tree Analysis and Debugging

Monte Carlo Tree Search (MCTS) planners build a search tree of belief and action nodes during planning. Inspecting this tree is essential for understanding planner behavior, tuning parameters, and diagnosing problems.

**Tools Covered:**
- `PolicyRunData` and `info_variables` — metrics returned by every `planner.action()` call
- `TreeMetrics` — enumeration of available tree statistics
- `compute_tree_metrics()` — compute statistics from a tree root node
- `print_tree()` — ASCII rendering of the search tree
- `plot_tree_graphs()` — interactive Plotly visualization

**Planners Demonstrated:** POMCP, PFT_DPW, SparsePFT

## 1. Accessing Tree Metrics

Every MCTS planner returns tree metrics via `PolicyRunData` when you call `planner.action(belief)`. These metrics are computed automatically from the search tree after each planning step.

In [1]:
import numpy as np
np.random.seed(42)

from POMDPPlanners.environments.tiger_pomdp import TigerPOMDP
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP
from POMDPPlanners.core.belief import get_initial_belief

env = TigerPOMDP(discount_factor=0.95)
planner = POMCP(
    environment=env,
    discount_factor=0.95,
    depth=10,
    exploration_constant=10.0,
    name="POMCP_Analysis",
    n_simulations=200,
)

belief = get_initial_belief(env, n_particles=200)
actions, run_data = planner.action(belief)

print(f"Selected action: {actions}")
print(f"Number of info variables: {len(run_data.info_variables)}")
print()

for var in run_data.info_variables:
    print(f"  {var.name}: {var.value}")

/home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-25 13:18:34,060 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing TigerPOMDP environment with discount factor 0.95
2026-02-25 13:18:34,060 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_Analysis (debug=False)


Selected action: ['listen']
Number of info variables: 7

  min_actions_visit_count: 1
  max_actions_visit_count: 197
  actions_visit_count_entropy: 0.09117633696120606
  n_actions_from_root: 3
  root_visit_count: 200
  tree_max_depth: 20
  is_leaf: 0


## 2. Understanding Each Metric

| Metric | Description | Healthy Values |
|---|---|---|
| `min_actions_visit_count` | Least-visited action from the root | > 0 (all actions explored) |
| `max_actions_visit_count` | Most-visited action from the root | Dominant but not overwhelming |
| `actions_visit_count_entropy` | Shannon entropy of action visit distribution | Higher = more uniform exploration |
| `n_actions_from_root` | Number of distinct actions expanded at root | Should equal total action count |
| `root_visit_count` | Total visits to root node | Equals `n_simulations` |
| `tree_max_depth` | Deepest path in the tree | Should approach planner `depth` |
| `is_leaf` | Whether the root is a leaf (no children) | 0 (should always have children) |

## 3. Metrics Across Multiple Runs

Tree metrics vary between runs due to stochastic sampling. Collecting statistics over multiple planning calls gives a more stable picture.

In [2]:
np.random.seed(42)

n_runs = 5
metric_records = {}

for run_idx in range(n_runs):
    b = get_initial_belief(env, n_particles=200)
    _, rd = planner.action(b)
    for var in rd.info_variables:
        metric_records.setdefault(var.name, []).append(var.value)

print(f"{'Metric':<35} {'Mean':>10} {'Std':>10}")
print("-" * 58)
for name, values in metric_records.items():
    arr = np.array(values, dtype=float)
    print(f"{name:<35} {np.mean(arr):>10.2f} {np.std(arr):>10.2f}")

# Convergence indicator: how concentrated is the search?
max_visits = np.array(metric_records["max_actions_visit_count"], dtype=float)
min_visits = np.array(metric_records["min_actions_visit_count"], dtype=float)
ratios = max_visits / np.maximum(min_visits, 1)
print(f"\nVisit ratio (max/min): mean={np.mean(ratios):.1f}, std={np.std(ratios):.1f}")
print("Higher ratio = more concentrated search (planner is more confident).")

Metric                                    Mean        Std
----------------------------------------------------------
min_actions_visit_count                   1.00       0.00
max_actions_visit_count                 196.80       0.40
actions_visit_count_entropy               0.10       0.01
n_actions_from_root                       3.00       0.00
root_visit_count                        200.00       0.00
tree_max_depth                           20.00       0.00
is_leaf                                   0.00       0.00

Visit ratio (max/min): mean=196.8, std=0.4
Higher ratio = more concentrated search (planner is more confident).


## 4. Building and Inspecting the Tree Directly

For deeper inspection, we can access the tree structure directly via `planner._learn_tree(belief)`. This is a **private method** intended for educational and debugging purposes — the public API is `planner.action()` which returns metrics via `PolicyRunData`.

In [3]:
np.random.seed(42)

from POMDPPlanners.core.tree import BeliefNode, ActionNode

belief = get_initial_belief(env, n_particles=200)
tree = planner._learn_tree(belief)  # Private method — for debugging only

print(f"Root v_value: {tree.v_value:.4f}")
print(f"Root visit_count: {tree.visit_count}")
print(f"Number of action children: {len(tree.children)}")
print()

# Inspect each action node
print(f"{'Action':<15} {'Q-value':>10} {'Visits':>10} {'Obs Children':>15}")
print("-" * 55)
for child in tree.children:
    if isinstance(child, ActionNode):
        print(
            f"{str(child.action):<15} {child.q_value:>10.4f} "
            f"{child.visit_count:>10} {len(child.children):>15}"
        )

Root v_value: -56.1815
Root visit_count: 200
Number of action children: 3

Action             Q-value     Visits    Obs Children
-------------------------------------------------------
listen            -56.1815        197               2
open_left        -214.4327          1               1
open_right       -320.5179          1               1


## 5. ASCII Tree

For a quick textual overview, `print_tree()` renders the tree structure. We use a small tree to keep the output readable.

In [4]:
np.random.seed(42)

from POMDPPlanners.core.tree import print_tree

small_planner = POMCP(
    environment=env,
    discount_factor=0.95,
    depth=3,
    exploration_constant=10.0,
    name="POMCP_Small",
    n_simulations=20,
)

small_belief = get_initial_belief(env, n_particles=50)
small_tree = small_planner._learn_tree(small_belief)

print("Tree structure (n_simulations=20, depth=3):")
print()
print_tree(small_tree)

2026-02-25 13:18:34,418 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_Small (debug=False)


Tree structure (n_simulations=20, depth=3):

BeliefNode:
                observation: None
                v_value: -53.73409375
                visit_count: 20
                weight: 1.0
                lower_confidence_bound: 0.0
                upper_confidence_bound: 0.0
                depth: 0
├── ActionNode:
                action: listen
                q_value: -271.9875
                visit_count: 1
                immediate_cost: None
                immediate_reward: None
                lower_confidence_bound: 0.0
                upper_confidence_bound: 0.0
                depth: 1
│   └── BeliefNode:
                observation: hear_left
                v_value: 0.0
                visit_count: 1
                weight: 1.0
                lower_confidence_bound: 0.0
                upper_confidence_bound: 0.0
                depth: 2
│       ├── ActionNode:
                action: listen
                q_value: 0.0
                visit_count: 0
                immed

## 6. Visualizing the Search Tree

`plot_tree_graphs()` creates two interactive Plotly panels showing node values and visit counts. This gives a visual understanding of how the planner allocates its search budget.

**Note**: The interactive plot may not render in all notebook environments. In JupyterLab, it displays as an interactive Plotly figure.

In [5]:
np.random.seed(42)

from POMDPPlanners.utils.visualization.tree_plots import plot_tree_graphs

viz_planner = POMCP(
    environment=env,
    discount_factor=0.95,
    depth=5,
    exploration_constant=10.0,
    name="POMCP_Viz",
    n_simulations=100,
)

viz_belief = get_initial_belief(env, n_particles=200)
viz_tree = viz_planner._learn_tree(viz_belief)

# This creates two side-by-side Plotly subplots:
# Left: node values (Q-values for actions, V-values for beliefs)
# Right: visit counts showing search allocation
plot_tree_graphs(viz_tree)

2026-02-25 13:18:36,079 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_Viz (debug=False)


Total number of nodes: 216
Tree depth: 11
Number of leaf nodes: 115
Value range: [-277.491, 19.500]
Visit count range: [0, 100]


## 7. Comparing Planner Trees

Different MCTS algorithms build trees with different structures. Let's compare POMCP, PFT_DPW, and SparsePFT on the same Tiger problem.

In [6]:
np.random.seed(42)

from POMDPPlanners.planners.mcts_planners.pft_dpw import PFT_DPW
from POMDPPlanners.planners.mcts_planners.sparse_pft import SparsePFT
from POMDPPlanners.utils.action_samplers import DiscreteActionSampler
from POMDPPlanners.utils.tree_statistics import compute_tree_metrics

n_sims = 200

planners = {
    "POMCP": POMCP(
        environment=env, discount_factor=0.95, depth=10,
        exploration_constant=10.0, name="POMCP", n_simulations=n_sims,
    ),
    "PFT_DPW": PFT_DPW(
        environment=env, discount_factor=0.95, depth=10,
        name="PFT_DPW",
        action_sampler=DiscreteActionSampler(actions=env.get_actions()),
        n_simulations=n_sims, exploration_constant=10.0,
    ),
    "SparsePFT": SparsePFT(
        environment=env, discount_factor=0.95, gamma=0.95, depth=10,
        c_ucb=1.0, beta_ucb=2.0, belief_child_num=3,
        name="SparsePFT", n_simulations=n_sims,
    ),
}

print(f"{'Planner':<12} {'Depth':>8} {'Entropy':>10} {'Min Visit':>12} {'Max Visit':>12} {'Root Visit':>12}")
print("-" * 70)

for name, p in planners.items():
    b = get_initial_belief(env, n_particles=200)
    t = p._learn_tree(b)
    metrics = compute_tree_metrics(t)
    m_dict = {m.name: m.value for m in metrics}
    print(
        f"{name:<12} "
        f"{m_dict.get('tree_max_depth', 'N/A'):>8} "
        f"{m_dict.get('actions_visit_count_entropy', 0):>10.3f} "
        f"{m_dict.get('min_actions_visit_count', 0):>12} "
        f"{m_dict.get('max_actions_visit_count', 0):>12} "
        f"{m_dict.get('root_visit_count', 0):>12}"
    )

2026-02-25 13:18:37,327 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP (debug=False)
2026-02-25 13:18:37,328 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: PFT_DPW (debug=False)
2026-02-25 13:18:37,328 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: SparsePFT (debug=False)


Planner         Depth    Entropy    Min Visit    Max Visit   Root Visit
----------------------------------------------------------------------
POMCP              20      0.265            1          191          200
PFT_DPW             7      1.568           53           77          200
SparsePFT          12      1.132           20          142          200


## 8. Effect of n_simulations

More simulations generally produce deeper, more converged trees. Let's see how tree metrics change as we increase the simulation budget.

In [7]:
np.random.seed(42)

sim_counts = [50, 100, 200, 500]

print(f"{'n_sims':>8} {'Depth':>8} {'Entropy':>10} {'Min Visit':>12} {'Max Visit':>12}")
print("-" * 55)

for n_sim in sim_counts:
    p = POMCP(
        environment=env, discount_factor=0.95, depth=15,
        exploration_constant=10.0, name=f"POMCP_{n_sim}",
        n_simulations=n_sim,
    )
    b = get_initial_belief(env, n_particles=200)
    t = p._learn_tree(b)
    metrics = compute_tree_metrics(t)
    m_dict = {m.name: m.value for m in metrics}
    print(
        f"{n_sim:>8} "
        f"{m_dict.get('tree_max_depth', 'N/A'):>8} "
        f"{m_dict.get('actions_visit_count_entropy', 0):>10.3f} "
        f"{m_dict.get('min_actions_visit_count', 0):>12} "
        f"{m_dict.get('max_actions_visit_count', 0):>12}"
    )

2026-02-25 13:18:38,604 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_50 (debug=False)
2026-02-25 13:18:38,616 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_100 (debug=False)
2026-02-25 13:18:38,647 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_200 (debug=False)
2026-02-25 13:18:38,716 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_500 (debug=False)


  n_sims    Depth    Entropy    Min Visit    Max Visit
-------------------------------------------------------
      50       12      0.732            3           42
     100       30      0.163            1           97
     200       30      0.127            1          196
     500       30      0.042            1          497


## 9. Effect of exploration_constant

The exploration constant controls the balance between exploring new actions and exploiting the best-known action. A higher value encourages more uniform exploration.

In [8]:
np.random.seed(42)

exploration_values = [1.0, 10.0, 50.0, 200.0]

print(f"{'Exploration':>12} {'Entropy':>10} {'Min Visit':>12} {'Max Visit':>12} {'Ratio':>8}")
print("-" * 58)

for exp_c in exploration_values:
    p = POMCP(
        environment=env, discount_factor=0.95, depth=10,
        exploration_constant=exp_c, name=f"POMCP_exp{exp_c}",
        n_simulations=200,
    )
    b = get_initial_belief(env, n_particles=200)
    t = p._learn_tree(b)
    metrics = compute_tree_metrics(t)
    m_dict = {m.name: m.value for m in metrics}
    
    min_v = max(float(m_dict.get('min_actions_visit_count', 0)), 1)
    max_v = float(m_dict.get('max_actions_visit_count', 0))
    ratio = max_v / min_v
    
    print(
        f"{exp_c:>12.1f} "
        f"{m_dict.get('actions_visit_count_entropy', 0):>10.3f} "
        f"{m_dict.get('min_actions_visit_count', 0):>12} "
        f"{m_dict.get('max_actions_visit_count', 0):>12} "
        f"{ratio:>8.1f}"
    )

print()
print("Higher exploration constant -> higher entropy -> more uniform visit distribution.")

2026-02-25 13:18:39,685 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_exp1.0 (debug=False)
2026-02-25 13:18:39,733 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_exp10.0 (debug=False)
2026-02-25 13:18:39,783 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_exp50.0 (debug=False)
2026-02-25 13:18:39,834 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_exp200.0 (debug=False)


 Exploration    Entropy    Min Visit    Max Visit    Ratio
----------------------------------------------------------
         1.0      0.127            1          196    196.0
        10.0      0.409            2          185     92.5
        50.0      0.187            1          194    194.0
       200.0      0.649           10          175     17.5

Higher exploration constant -> higher entropy -> more uniform visit distribution.


## 10. Diagnosing Common Issues

| Symptom | Likely Cause | Fix |
|---|---|---|
| Low entropy, high visit ratio | Premature convergence | Increase `exploration_constant` |
| Shallow `tree_max_depth` | Too few simulations or low depth | Increase `n_simulations` or `depth` |
| All visits nearly equal | Over-exploration, no convergence | Decrease `exploration_constant` |
| `is_leaf = 1` | Root has no children | Increase `n_simulations`; check belief validity |
| `n_actions_from_root < total_actions` | Not all actions explored | Increase `n_simulations` or `exploration_constant` |

Let's demonstrate a deliberately bad configuration and compare it to a good one.

In [9]:
np.random.seed(42)

# Bad config: very low exploration, very few simulations
bad_planner = POMCP(
    environment=env, discount_factor=0.95, depth=3,
    exploration_constant=0.001, name="POMCP_Bad",
    n_simulations=20,
)

# Good config: balanced exploration, adequate simulations
good_planner = POMCP(
    environment=env, discount_factor=0.95, depth=10,
    exploration_constant=10.0, name="POMCP_Good",
    n_simulations=200,
)

configs = {"Bad (exp=0.001, sims=20)": bad_planner, "Good (exp=10, sims=200)": good_planner}

print(f"{'Config':<30} {'Depth':>8} {'Entropy':>10} {'Min':>6} {'Max':>6} {'Leaf':>6}")
print("-" * 70)

for label, p in configs.items():
    b = get_initial_belief(env, n_particles=200)
    t = p._learn_tree(b)
    metrics = compute_tree_metrics(t)
    m_dict = {m.name: m.value for m in metrics}
    print(
        f"{label:<30} "
        f"{m_dict.get('tree_max_depth', 'N/A'):>8} "
        f"{m_dict.get('actions_visit_count_entropy', 0):>10.3f} "
        f"{m_dict.get('min_actions_visit_count', 0):>6} "
        f"{m_dict.get('max_actions_visit_count', 0):>6} "
        f"{m_dict.get('is_leaf', 0):>6}"
    )

print()
print("The bad config produces a shallow, poorly-explored tree.")
print("The good config explores all actions with a deeper, more balanced tree.")

2026-02-25 13:18:40,840 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_Bad (debug=False)
2026-02-25 13:18:40,842 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_Good (debug=False)


Config                            Depth    Entropy    Min    Max   Leaf
----------------------------------------------------------------------
Bad (exp=0.001, sims=20)              6      0.591      1     17      0
Good (exp=10, sims=200)              20      0.091      1    197      0

The bad config produces a shallow, poorly-explored tree.
The good config explores all actions with a deeper, more balanced tree.


## Summary: Debugging Workflow

When a planner is not performing well, follow this diagnostic process:

1. **Check tree metrics** — use `planner.action(belief)` and inspect `run_data.info_variables`
2. **Verify `is_leaf`** — if 1, the tree has no children (increase simulations or check belief)
3. **Check `n_actions_from_root`** — should equal the number of available actions
4. **Look at entropy and visit ratio** — low entropy may indicate premature convergence
5. **Inspect `tree_max_depth`** — shallow trees suggest insufficient simulation budget
6. **Compare across configs** — sweep `exploration_constant` and `n_simulations` to find good values
7. **Use `print_tree()` or `plot_tree_graphs()`** — visual inspection often reveals structural issues

## What's Next?

- **Basic usage**: See [basic_usage.ipynb](basic_usage.ipynb) for a full introduction
- **Belief representations**: See [belief_representations.ipynb](belief_representations.ipynb) for particle, Gaussian, and GMM beliefs
- **Build your own environment**: See [custom_environment.ipynb](custom_environment.ipynb) to create a POMDP from scratch
- **Compare planners at scale**: See [planners_comparison.ipynb](planners_comparison.ipynb) for batch evaluations with statistical analysis
- **Tune hyperparameters**: See [hyperparameter_tuning.ipynb](hyperparameter_tuning.ipynb) for optimizing planner parameters with Optuna